# 03 — Análisis y particiones

Valida los JSONL, estudia su distribución y crea particiones agrupadas por documento objetivo. Un mismo documento nunca debe aparecer simultáneamente en entrenamiento y evaluación.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif ROOT.name == "aulaTEX":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if (ROOT / "aulaTEX/aulatex_training").is_dir():
    from aulaTEX.aulatex_training.motor_training import assert_no_group_leakage, dataset_stats, read_jsonl
else:
    from scripts.aulatex_training.motor_training import assert_no_group_leakage, dataset_stats, read_jsonl

data_root = ROOT / "aulaTEX/data" if (ROOT / "aulaTEX/data").is_dir() else ROOT / "data"
split_dir = data_root / "private/splits"
splits = {name: read_jsonl(split_dir / f"sft-{name}.jsonl") for name in ("train", "validation", "test")}
assert_no_group_leakage(splits)
for name, rows in splits.items():
    print(name, json.dumps(dataset_stats(rows), ensure_ascii=False))
print(json.loads((split_dir / "manifest.json").read_text(encoding="utf-8")))